# Demo: symulacja dostaw pomocy humanitarnej w Strefie Gazy

Ten notatnik pokazuje po kolei, co robi symulacja. Każdy krok ma krótki opis i komórkę do uruchomienia.

**Co symuluje model**

- Ciężarówki z pomocą wjeżdżają do Gazy przez przejścia graniczne (głównie Kerem Shalom).
- Na przejściu czekają w kolejce i są odprawiane. Przejście ma ograniczoną pojemność K.
- Potem jadą po drogach z OpenStreetMap do jednego z pięciu hubów. Zniszczone drogi jadą wolniej.
- W hubie rozładowują ładunek. Ludzie w hubie zjadają żywność, a model liczy, ile potrzeb udało się pokryć.

**Skąd dane**

- Liczba ciężarówek, przejście, ładunek i odbiorca: pliki ONZ (`Commodities-Received`, `Commodities-Dispatching`).
- Zniszczenia dróg: UNOSAT. Sieć dróg: OpenStreetMap.
- Czas odprawy, liczba stanowisk i pojemność K nie są w danych. Wzięliśmy je z artykułów albo je założyliśmy (patrz README, sekcja „Źródła”).

**Czas uruchomienia:** cały notatnik to ok. 10 minut. Kroki 6 i 7 są najdłuższe (kilka symulacji pod rząd).

## Krok 0. Przygotowanie

Komórka znajduje katalog projektu i ustawia miesiąc. Domyślnie to **maj 2024**: działają wtedy cztery przejścia, a ładunek trafia do kilku hubów.
Inny miesiąc ustawisz w zmiennej `MIESIAC` (np. `"2024-03"` albo `"2023-11"`).

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Katalog projektu: ten, w którym leży src/run.py
ROOT = Path.cwd()
if not (ROOT / "src" / "run.py").exists():
    ROOT = ROOT / "ssd_projekt"
if not (ROOT / "src" / "run.py").exists():
    raise FileNotFoundError("Uruchom notatnik z katalogu ssd_projekt")
sys.path.insert(0, str(ROOT / "src"))
%cd {ROOT}

MIESIAC = "2024-05"        # miesiąc z danych ONZ (paź 2023 - sty 2025)
LICZBA_ZIAREN = 5          # ile przebiegów w kroku 7
OUT = ROOT / "output" / MIESIAC


def uruchom(*argumenty, pokaz=False):
    """Uruchamia src/run.py z podanymi flagami. Wynik pokazuje tylko, gdy pokaz=True."""
    wynik = subprocess.run([sys.executable, "src/run.py", "--month", MIESIAC, *argumenty],
                           capture_output=True, text=True)
    if wynik.returncode != 0:
        print(wynik.stderr[-2000:])
        raise RuntimeError("symulacja zakończyła się błędem")
    if pokaz:
        print(wynik.stdout)
    return wynik.stdout


plt.rcParams["figure.figsize"] = (11, 4)
print("Katalog projektu:", ROOT, "| miesiąc:", MIESIAC)

## Krok 1. Dane: ile ciężarówek i dokąd jadą

Z pliku `Commodities-Received` bierzemy ciężarówki z wybranego miesiąca: jedna ciężarówka to jeden wiersz.
Z pliku `Commodities-Dispatching` liczymy, jaki **udział wydań** trafia do każdego z pięciu hubów w tym miesiącu.
Z tego udziału losujemy cel każdej ciężarówki.
Ludność każdego huba bierzemy osobno, z raportów IPC (liczba mieszkańców gubernatorstwa w danym miesiącu). Nie zależy od tego, ile pomocy tam jedzie.

Na wykresie: ile ciężarówek przyjechało danego dnia i którym przejściem.

In [ ]:
import data

ciezarowki, udzialy = data.prepare(MIESIAC)
print(f"Ciężarówek w miesiącu: {len(ciezarowki)}")
print(ciezarowki["crossing"].value_counts().to_string())

dzienne = ciezarowki.groupby(["date", "crossing"]).size().unstack(fill_value=0)
dzienne.index = dzienne.index.day
dzienne.plot.bar(stacked=True, width=0.85)
plt.xlabel("dzień miesiąca"); plt.ylabel("ciężarówek"); plt.title(f"Ciężarówki wjeżdżające do Gazy, {MIESIAC}")
plt.show()

huby = pd.DataFrame({"udział wydań": udzialy.iloc[0].round(3),
                     "ludność huba": pd.Series(data.hub_population(MIESIAC))}).fillna(0).astype({"ludność huba": int})
print("\nCel ciężarówek (z wydań) i ludność hubów (z IPC):")
display(huby)

## Krok 2. Mapa dróg i zniszczeń

Model jeździ po prawdziwej sieci dróg Gazy z OpenStreetMap. Zniszczenia z UNOSAT są liczone w kratkach 500 m.
Im więcej zniszczeń w kratce, tym wolniej jedzie się po drodze (połowa prędkości przy 150 punktach).

Poniżej dwie mapy:

- **diagnoza.png**: przejścia, huby i najkrótsze trasy przejście → hub na jednej mapie (kolor = przejście, na wspólnych odcinkach linie biegną obok siebie),
- **mapa_zniszczen.png**: drogi pomalowane według tego, jak bardzo są zwolnione.

To uruchomienie niczego nie symuluje. Tylko buduje graf i trasy.

In [ ]:
uruchom("--diagnose")
for plik in ["diagnoza.png", "mapa_zniszczen.png"]:
    display(Image(str(OUT / plik)))

## Krok 3. Podstawowa symulacja: kolejka na przejściach i dojazd do hubów

To jest rdzeń modelu. Każda ciężarówka przechodzi kolejne stany:

1. **wjazd** na przejście (dzień z danych, godzina losowana między 8 a 18),
2. **kolejka**, jeśli wszystkie stanowiska są zajęte,
3. **odprawa** (średnio 45 minut),
4. **jazda** po grafie do swojego huba, z trasą liczoną algorytmem D* Lite,
5. **dowiozła** ładunek.

Tabele pokazują średnie, 95. percentyl i maksimum czasu w kolejce (w minutach) oraz czasy dojazdu do hubów.

In [ ]:
uruchom("--log-days", "1", "--log-every", "300")   # log tekstowy: wszystkie ciężarówki z 1. dnia
print("Kolejki na przejściach [min]:")
display(pd.read_csv(OUT / "summary_crossings.csv"))
print("Dojazd do hubów (tylko dowiezione ciężarówki):")
display(pd.read_csv(OUT / "summary_hubs.csv"))

log = pd.read_csv(OUT / "truck_log.csv")
for przejscie, g in log.groupby("crossing"):
    if len(g) > 150:
        plt.hist(g["queue_min"].dropna() / 60, bins=40, alpha=0.6, label=przejscie)
plt.xlabel("czas w kolejce [godziny]"); plt.ylabel("liczba ciężarówek")
plt.title("Ile godzin ciężarówki czekają na odprawę"); plt.legend(); plt.show()

## Krok 4. Życie jednej ciężarówki

Symulacja zapisuje dziennik zdarzeń. Poniżej dziennik jednej ciężarówki od wjazdu do dojazdu:
na którym przejściu stała, ile ludzi czekało przed nią, ile trwała odprawa, jaką trasą jechała i dokąd dojechała.

In [ ]:
import re

linie = (OUT / "sim_log.txt").read_text(encoding="utf-8").splitlines()
numery = [m.group(1) for l in linie if (m := re.search(r"\] #(\d+) ", l))]
wybrana = numery[len(numery) // 2]     # ciężarówka ze środka dnia
print(f"Ciężarówka #{wybrana}:\n")
for l in linie:
    if f"#{wybrana} " in l:
        print(l)

## Krok 5. Ile ciężarówek czeka w kolejce w ciągu miesiąca

Wykres pokazuje, ile ciężarówek naraz czeka w kolejce na przejściu Kerem Shalom (nie licząc tych, które są już w odprawie).
Wysokie szczyty to dni, w których przyjechało więcej ciężarówek, niż stanowiska zdążą odprawić.
Kolejka z poprzedniego dnia przechodzi na następny, bo odprawa trwa także po zamknięciu bramy.

In [ ]:
ks = log[(log["crossing"] == "Kerem Shalom") & log["queue_min"].notna()]
wjazdy = pd.Series(1, index=ks["arrival_min"].to_numpy())
starty = pd.Series(-1, index=(ks["arrival_min"] + ks["queue_min"]).to_numpy())
w_kolejce = pd.concat([wjazdy, starty]).sort_index().cumsum()

plt.step(w_kolejce.index / 1440 + 1, w_kolejce.values, where="post")
plt.xlabel("dzień miesiąca"); plt.ylabel("ciężarówek w kolejce")
plt.title("Kerem Shalom: długość kolejki w czasie")
plt.show()
print(f"Największa kolejka: {w_kolejce.max()} ciężarówek. Pojemność K przejścia to 200 (odprawa + kolejka).")

## Krok 6. Pełny model: rozładunek, żywność i zdarzenia losowe

Teraz włączamy dodatkowe warstwy (flaga `--m2`):

- **Rozładunek w hubie.** W hubie jest kilka doków. Ciężarówka czeka na wolny dok i rozładowuje się ok. 30 minut.
- **Popyt na żywność.** Każdy hub ma ludność (z IPC) i zapas żywności. Co dobę ludzie zjadają tyle, ile jest w zapasie (do 2100 kcal na osobę).
  Z kalorii wychodzi uproszczona faza bezpieczeństwa żywnościowego od 1 (dobrze) do 5 (katastrofa).
  To **nie jest** oficjalny wskaźnik IPC.
- **Zdarzenia losowe.** Incydenty na drogach, zamknięcia przejść i awarie ciężarówek. Częstości są przykładowe, nie skalibrowane.

Uwaga: ta komórka nadpisuje `truck_log.csv` z kroku 3.

In [ ]:
uruchom("--m2")
print("Pokrycie zapotrzebowania na żywność w hubach:")
display(pd.read_csv(OUT / "demand_summary.csv"))
print("Najważniejsze wskaźniki:")
display(pd.read_csv(OUT / "kpi_summary.csv"))

**Wykresy z pełnego modelu**

- **kpi_supply_coverage**: zapas żywności i kalorie na osobę w hubach w czasie,
- **kpi_ipc_timeline**: uproszczona faza bezpieczeństwa żywnościowego w czasie,
- **kpi_queue_dist**: rozkład czasów kolejki,
- **kpi_delivery_timeline**: ile ciężarówek dowieziono każdego dnia,
- **kpi_congestion_map**: które drogi były najbardziej obciążone.

In [ ]:
for plik in ["kpi_supply_coverage.png", "kpi_ipc_timeline.png", "kpi_queue_dist.png",
             "kpi_delivery_timeline.png", "kpi_congestion_map.png"]:
    if (OUT / plik).exists():
        display(Image(str(OUT / plik)))

## Krok 7. Scenariusze: co się zmieni, gdy zmienimy jedną rzecz

Ten sam model, pięć wariantów:

| Scenariusz | Co zmieniamy |
|---|---|
| A | nic (punkt odniesienia) |
| B | odprawa dwa razy szybsza |
| C | zamknięcie najruchliwszej drogi |
| D | dwa razy więcej doków w hubach |
| E | dwa razy więcej ciężarówek przy tej samej przepustowości granicy |

W tabeli: kolejka na Kerem Shalom w godzinach, ile ciężarówek odrzucono przez pojemność K i jaką część potrzeb żywnościowych pokryto.
Scenariusz C zamyka jedną drogę, a objazd jest bardzo krótki, więc prawie nic nie zmienia.
Komórka uruchamia pięć symulacji, więc trwa kilka minut.

In [ ]:
OPISY = {"A": "punkt odniesienia", "B": "odprawa 2x szybsza", "C": "zamknięta najruchliwsza droga",
         "D": "2x więcej doków", "E": "2x więcej ciężarówek"}
wiersze = []
for lit in "ABCDE":
    uruchom("--m2", "--scenario", lit)
    kat = OUT / f"scenario_{lit}"
    l = pd.read_csv(kat / "truck_log.csv")
    d = pd.read_csv(kat / "demand_summary.csv")
    ks = l[l["crossing"] == "Kerem Shalom"]["queue_min"].dropna() / 60
    wiersze.append({
        "scenariusz": f"{lit}: {OPISY[lit]}",
        "ciężarówek": len(l),
        "kolejka KS średnia [h]": round(ks.mean(), 1),
        "kolejka KS p95 [h]": round(ks.quantile(0.95), 1),
        "odrzuconych (K)": int((l["final_state"] == "odrzucona").sum()),
        "dowiezionych [%]": round(100 * l["delivered"].mean(), 1),
        "pokrycie żywności [%]": round(100 * d["food_consumed_mt"].sum() / d["food_need_mt"].sum(), 1),
    })
porownanie = pd.DataFrame(wiersze).set_index("scenariusz")
display(porownanie)

fig, osie = plt.subplots(1, 2, figsize=(12, 4))
porownanie["kolejka KS średnia [h]"].plot.bar(ax=osie[0], color="tab:blue", rot=20)
osie[0].set_title("Średnia kolejka na Kerem Shalom [h]"); osie[0].set_xlabel("")
porownanie["pokrycie żywności [%]"].plot.bar(ax=osie[1], color="tab:green", rot=20)
osie[1].set_title("Pokrycie zapotrzebowania na żywność [%]"); osie[1].set_xlabel("")
plt.tight_layout(); plt.show()

## Krok 8. Czy wynik zależy od szczęścia? Wiele losowań

Symulacja losuje godzinę wjazdu, czas odprawy, cel ciężarówki i zdarzenia losowe.
Jeden przebieg może więc wyjść trochę inaczej niż drugi. Dlatego uruchamiamy ją kilka razy z różnymi ziarnami losowania
i podajemy **średnią i odchylenie standardowe**. Małe odchylenie oznacza, że wynik jest stabilny.

In [ ]:
uruchom("--m2", "--multi-seed", str(LICZBA_ZIAREN))
print(f"Średnia i odchylenie z {LICZBA_ZIAREN} przebiegów:")
display(pd.read_csv(OUT / "multiseed_summary.csv", index_col=0))

przebiegi = pd.read_csv(OUT / "multiseed_kpi.csv")
przebiegi.plot.bar(x="seed", y="queue_mean_min", legend=False, rot=0)
plt.ylabel("średnia kolejka [min]"); plt.title("Średnia kolejka w kolejnych przebiegach"); plt.show()

## Podsumowanie i ograniczenia

**Co model już pokazuje**

- Kolejka na przejściach jest rzędu godzin, a przy dwa razy większym ruchu rośnie do kilkunastu godzin i część ciężarówek jest odrzucana.
- Szybsza odprawa skraca kolejkę najbardziej ze wszystkich zmian.
- Cel ciężarówek, ludność hubów i zniszczenia dróg pochodzą z danych dla wybranego miesiąca.

**Czego nie wiemy i na co trzeba uważać**

- Liczba stanowisk odprawy i pojemność K są oszacowane, nie zmierzone. Wyniki kolejki są poprawne co do rzędu wielkości.
- Czekanie przed dniem z danych (np. kilka tygodni po stronie egipskiej) nie jest w modelu.
- Ludność hubów pochodzi z IPC, ale IPC łączy gubernatorstwa, więc ich podział na pojedyncze huby to nasze założenie. Zapas nie przepływa między hubami, więc nadwyżka w jednym nie pomaga drugiemu.
- Faza bezpieczeństwa żywnościowego jest uproszczona (tylko kalorie). Wody model nie uwzględnia, bo nie ma jej w danych.
- Zdarzenia losowe mają przykładowe częstości.

Pełny opis założeń, tabele i źródła są w `README.md`.